In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!ls /content/drive/MyDrive

Mounted at /content/drive
 20191216_122821.mp4		  gptneo-finetuned-qa
 20200101_074036.mp4		  math
 20200324_211649.mp4		 'Screenshot_20220307-174728_().jpg'
 Classroom			  대마도여행250224.gmap
'Colab Notebooks'		  여수여행_241216.gmap
 Colab_Notebooks		 '역사 보고서.show'
'Gantt Chart_07의 사본.gslides'   오키나와여행_241230.gmap
 gptneo-1.3B-finetuned		  통지서.html
 gptneo-1.3B-masked		  홍콩여행_240121.gmap


In [2]:
# 1. 필요한 패키지 설치
!pip install transformers datasets accelerate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# 2. 모델 로딩
from transformers import GPTNeoForCausalLM, GPT2TokenizerFast
import torch

model_name = "EleutherAI/gpt-neo-1.3B"
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT 모델엔 필수
model = GPTNeoForCausalLM.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.31G [00:00<?, ?B/s]

In [ ]:
model.gradient_checkpointing_enable()

In [4]:
# 3. 문제 데이터
from google.colab import files
import json

uploaded = files.upload()

with open("augmented_general_500.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

Saving augmented_general_500.json to augmented_general_500.json


In [5]:
# 4. 문제+답 텍스트 생성
formatted_data = []
for item in raw_data:
    prompt = f"Question: {item['Question']}\nAnswer: {item['Answer']}"
    formatted_data.append({"Text": prompt})

In [6]:
# 5. 마스킹 전처리 함수
def tokenize_and_mask(example):
    text = example["Text"]

    # 'Answer:' 시작 위치 찾기
    answer_index = text.find("Answer:")
    if answer_index == -1:
        answer_index = 0

    # 토크나이징 (offset_mapping 사용)
    tokenized = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=384,
        return_offsets_mapping=True
    )

    input_ids = tokenized["input_ids"]
    offsets = tokenized["offset_mapping"]

    labels = input_ids[:]  # 복사

    # 'Answer:' 이전은 학습에서 무시
    for i, (start, _) in enumerate(offsets):
        if start < answer_index:
            labels[i] = -100

    tokenized["labels"] = labels
    tokenized.pop("offset_mapping")  # 필요 없는 필드 제거
    return tokenized

In [7]:
# 6. Dataset 변환 및 전처리
from datasets import Dataset

dataset = Dataset.from_list(formatted_data)
tokenized_dataset = dataset.map(tokenize_and_mask, batched=False)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
# 7. Trainer 준비
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./gptneo-finetuned-masked",
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    logging_steps=10,
    save_steps=250,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
)

In [ ]:
# 8. Trainer 구성 & 학습 실행
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

<ipython-input-11-ceccad0ce5d7>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: circlehalf17 (circlehalf17-no-job) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
10,1.367200
20,1.156600
30,0.854000
40,0.753400
50,0.653800
60,0.599400
70,0.646800
80,0.593000
90,0.546300
100,0.610000


TrainOutput(global_step=1000, training_loss=0.2287413665652275, metrics={'train_runtime': 611.3831, 'train_samples_per_second': 6.543, 'train_steps_per_second': 1.636, 'total_flos': 1.1137122828288e+16, 'train_loss': 0.2287413665652275, 'epoch': 10.0})

In [ ]:
model.save_pretrained("/content/drive/MyDrive/gptneo-1.3B-masked")
tokenizer.save_pretrained("/content/drive/MyDrive/gptneo-1.3B-masked")

('/content/drive/MyDrive/gptneo-1.3B-masked/tokenizer_config.json',
 '/content/drive/MyDrive/gptneo-1.3B-masked/special_tokens_map.json',
 '/content/drive/MyDrive/gptneo-1.3B-masked/vocab.json',
 '/content/drive/MyDrive/gptneo-1.3B-masked/merges.txt',
 '/content/drive/MyDrive/gptneo-1.3B-masked/added_tokens.json',
 '/content/drive/MyDrive/gptneo-1.3B-masked/tokenizer.json')